# 67 — A/B Prompt Testing
**Goal:** Quantitatively compare prompt variants on a golden test set.

Prompt engineering without measurement is just editing. This chapter builds a small **A/B testing framework**: a fixed **golden set** of (input, expected-output) pairs, a harness that runs any prompt variant over that set, and a score that says which variant is better. It then adds the statistical machinery needed to decide whether an observed score difference is real or just noise.

**Why it matters for resumes / ATS:** the wording of a rewriting or extraction prompt is a product decision — "Developed" vs "Created" changes how a candidate is presented to employers. A/B testing makes that decision evidence-based: instead of arguing about prompt style, you run both variants on a golden set of representative resume bullets and let the scores decide. The golden set also becomes a regression net: any future prompt edit that drops its score is caught immediately — the evaluation equivalent of a unit test.

## 1. A/B Testing Framework

The `PromptABTest` class is the whole loop in 20 lines: hold a golden set of `(input, expected_output)` pairs, run a prompt variant over every pair, and mark each output `exact` only if it matches the expected string byte-for-byte after stripping. Accuracy is the fraction of exact matches. The design deliberately separates the *prompt template* from the *model function* (`llm_fn`), so the same harness measures a template change, a model change, or a system-prompt change without touching the golden set.

**What the code does:** the golden set holds three weak bullets ("Responsible for ML models") paired with strong impact-phrased rewrites ("Developed ML models achieving 95% accuracy"). Variant A swaps in aggressive verbs (`Developed`, `Built`, `Led`); variant B uses conservative ones (`Created`, `Designed`, `Managed`). Running the harness reports **0% accuracy for both variants**: neither produces the expected strings, because the gold rewrites add impact phrases ("achieving 95% accuracy") that simple verb substitution cannot generate. The framework is working as intended — exact match is brutally strict, and the result forces you to align the metric with the task's real success criterion.

**Try it:** relax the comparison to "expected verb appears in the output" and both variants jump — the metric should encode what you actually care about.

In [ ]:
class PromptABTest:
    def __init__(self, golden_set):
        self.golden = golden_set  # [(input, expected_output), ...]
    
    def evaluate(self, prompt_template, llm_fn):
        """Test a prompt variant against golden set."""
        results = []
        for input_text, expected in self.golden:
            output = llm_fn(prompt_template.format(input=input_text))
            exact = output.strip() == expected.strip()
            results.append({"input": input_text, "expected": expected, "got": output, "exact": exact})
        accuracy = sum(r["exact"] for r in results) / len(results)
        return {"accuracy": accuracy, "results": results}

# Simulated golden set
golden = [
    ("Responsible for ML models", "Developed ML models achieving 95% accuracy"),
    ("Was in charge of data pipeline", "Built automated data pipeline reducing processing time by 60%"),
    ("Helped with team projects", "Led cross-functional team delivering 3 major features"),
]

# Simulate A/B test
def variant_a(text):
    return text.replace("Responsible for", "Developed").replace("Was in charge of", "Built").replace("Helped with", "Led")

def variant_b(text):
    return text.replace("Responsible for", "Created").replace("Was in charge of", "Designed").replace("Helped with", "Managed")

test = PromptABTest(golden)
print("A/B Test Results:")
for name, fn in [("Variant A (aggressive verbs)", variant_a), ("Variant B (conservative)", variant_b)]:
    result = test.evaluate("Rewrite: {input}", fn)
    print(f"  {name}: {result['accuracy']:.0%} accuracy on golden set")
    for r in result['results']:
        if not r['exact']:
            print(f"    ✗ '{r['input'][:30]}...' -> '{r['got'][:30]}...' (expected '{r['expected'][:30]}...')")

## 2. Statistical Significance

One run per variant proves nothing: LLM outputs are stochastic, so a single golden-set score is a sample, not a measurement. The standard fix is to repeat the evaluation several times and ask whether the two score distributions differ more than chance would predict. The **independent two-sample t-test** compares the means while accounting for variance; the **p-value** is the probability of seeing a difference this large if the variants were actually identical.

**What the code does:** ten score observations per variant — A clustered near 0.859, B near 0.823. The test returns a t-statistic of 7.453 and p ≈ 0.0000 (below 0.0001), so the branch prints "Statistically significant (p<0.05)". With tightly clustered scores the conclusion is robust: A is genuinely better, not lucky. The lesson applies in reverse too — with noisy real-world evaluations and n=10, many observed gaps will *not* be significant, and the honest answer is "keep collecting data".

**Try it:** nudge two of B's scores up to 0.86 and the p-value jumps — small samples turn real differences into coin flips.

In [ ]:
from scipy import stats
import numpy as np

# Example: 10 runs each of two prompt variants
variant_a_scores = [0.85, 0.87, 0.86, 0.84, 0.88, 0.85, 0.86, 0.87, 0.85, 0.86]
variant_b_scores = [0.82, 0.83, 0.81, 0.84, 0.82, 0.83, 0.81, 0.82, 0.83, 0.82]

t_stat, p_value = stats.ttest_ind(variant_a_scores, variant_b_scores)
print(f"A/B Statistical Test:")
print(f"  Variant A mean: {np.mean(variant_a_scores):.3f}")
print(f"  Variant B mean: {np.mean(variant_b_scores):.3f}")
print(f"  t-statistic: {t_stat:.3f}")
print(f"  p-value: {p_value:.4f}")
print(f"  {'✓ Statistically significant (p<0.05)' if p_value < 0.05 else 'Not significant — difference may be noise'}")

## Summary: A/B test prompts against golden datasets. Use statistical tests to confirm significance.

**Prompt changes should be adopted on evidence, not vibes — and evidence means repeated runs, not one score.**

The golden set turns prompt engineering into a measurable loop, and the framework's strict exact-match scoring shows why the metric must be designed together with the task: the right test for a rewriting prompt is not byte equality but "did the rewrite do the job." Statistical testing separates real gains from sampling noise, and the same golden set doubles as a regression net for future prompt edits. What this chapter cannot score is *quality* — whether "Developed ML models achieving 95% accuracy" is a true, defensible claim. That judgment requires humans, and it is the subject of the next chapter: human evaluation protocols.